# College Football Visualization Challenge 🏈

Welcome! Your goal is to make **one chart that helps answer a football question**. You do not need to know Python or statistics before starting. Run each gray code box by clicking its **▶** button, from top to bottom. The notes tell you what each step does.

> **Your only job:** follow the steps, choose something you are curious about, and make the chart your own. You do not need to memorize the code or learn everything in this notebook today.

### A few questions you could explore

- Do run plays and pass plays gain different amounts of yardage?
- Do teams gain more yards on some downs than others?
- Does the play type that works best change by quarter?
- Which teams have the most positive average expected-points-added (EPA) per play?
- What happens to a team's play choices when it is ahead or behind?

You can use one of these questions or make up your own. A good chart does not need to be complicated.

## Step 1: Connect to the event data

This notebook reads a CSV file from the event's public GitHub repository.

In [ ]:
# The organizer should replace YOUR-USERNAME and YOUR-REPOSITORY in this link.
DATA_URL = "https://raw.githubusercontent.com/alliemcd/SAA_football/main/data/college_football_plays_2025.csv"

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the charts readable. You can ignore these style settings.
sns.set_theme(style="whitegrid", context="notebook")

# If the organizer has filled in the link, read the CSV from GitHub.
# Otherwise, let you choose the CSV file from your computer.
if "alliemcd" not in DATA_URL and "SAA_football" not in DATA_URL:
    plays = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Choose the college_football_plays_2025.csv file.")
        uploaded_files = files.upload()
        uploaded_name = next(iter(uploaded_files))
        plays = pd.read_csv(uploaded_name)
    except ImportError:
        # This option is for someone running Jupyter on their own computer.
        plays = pd.read_csv("data/college_football_plays_2025.csv")

print("Data loaded!")

### What is this data?

Each **row** is one offensive play. Each **column** describes something about that play, such as the team, the quarter, or yards gained. The collection script leaves out extra source columns so we can focus on a few useful details.

A blank value means the source did not have that detail for that play. You can still make charts without using every column.

In [ ]:
# See the first few plays, just to get a feel for how the table is laid out.
plays.head()

### Learn the column names

The left side of the table shows each column name. The right side shows what kind of information it holds. For example, `yards_gained` is how many yards were recorded on the play.

In [ ]:
# This shows how many rows and columns are in the table, then lists the column names.
print(f"There are {plays.shape[0]:,} plays and {plays.shape[1]} columns.")
print("\nColumns:")
for column_name in plays.columns:
    print("•", column_name)

| Column | In everyday words |
|---|---|
| `season`, `week`, `game_id` | Which season, week, and game. |
| `offense_team`, `defense_team` | The team with the ball and the team defending. |
| `quarter` | The quarter when the play began. |
| `play_type` | Whether it was a run, pass, or another kind of play from scrimmage. |
| `down` | First, second, third, or fourth down. |
| `yards_to_go` | How many yards the offense needed for a first down. |
| `yards_gained` | Yardage recorded for the play. |
| `epa` | Expected points added: an estimate of how the play changed the offense's scoring expectation. Positive is usually better for the offense; negative is usually worse. |
| `successful` | Whether the source's EPA model counted the play as successful. |
| `score_difference_at_start` | The offense's score minus the defense's score before the play. A positive number means the offense was ahead. |

You do not need to understand every column. Pick a question that sounds interesting and use only the columns that help answer it.

## Step 2: Make a first chart together

Let's ask: **On average, how many yards were gained on runs and passes?**

A bar chart is useful for comparing groups. This code first calculates the average yards gained for each play type, then draws one bar for each type. An average is a way to summarize a group with one typical value.

In [ ]:
# Calculate the average yards gained for each type of play.
average_yards = (
    plays.dropna(subset=["play_type", "yards_gained"])
    .groupby("play_type", as_index=False)["yards_gained"]
    .mean()
)

# Draw the chart. You can ignore the details for now; the next sections show
# which small parts you can change.
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=average_yards, x="play_type", y="yards_gained", color="steelblue", ax=ax)
ax.set_title("Average Yards Gained by Play Type", fontsize=16)
ax.set_xlabel("Type of play")
ax.set_ylabel("Average yards gained")
plt.tight_layout()
plt.show()

### How to read this chart

- The labels along the bottom are the **play types** (the groups being compared).
- The vertical direction shows **average yards gained**. Taller bars mean a higher average.
- Use the chart to describe what you see. It shows a pattern in this dataset; by itself, it does not prove why the pattern happened.

Notice that averages can hide variation: two play types might have similar averages even if one has more very short and very long plays. That could be a question for another chart.

## Step 3: Change one thing and see what happens

The next chart compares **yards to go** with **yards gained**, and uses color to mark the play type. A dot is one play.

To customize it, change just the three choices below: `x_feature`, `y_feature`, and `color_feature`. The spelling must match a column name in the table. Keep the quotation marks.

Try these choices:
- For the horizontal axis: `yards_to_go`, `down`, `quarter`, `epa`, or `score_difference_at_start`
- For the vertical axis: `yards_gained` or `epa`
- For color: `play_type`, `offense_team`, `defense_team`, `down`, or `quarter`

For example, change `color_feature = "play_type"` to `color_feature = "quarter"` to color each dot by quarter. Then run this cell again.

In [ ]:
# Edit these quoted column names to customize your chart.
x_feature = "yards_to_go"
y_feature = "yards_gained"
color_feature = "play_type"

# You may also type a team name from the list below. Leave this as "All teams"
# to use every team's plays.
team_to_show = "All teams"

print("Some offense team names:")
print(plays["offense_team"].dropna().drop_duplicates().sort_values().head(25).to_list())

Now draw the chart using your choices. If you typed a team name, the chart will show only that team's offense. We use a random sample of up to 4,000 plays so the chart loads quickly and the dots are easier to see.

In [ ]:
# Start with all plays, or keep only the team you chose above.
chart_data = plays.copy()
if team_to_show != "All teams":
    chart_data = chart_data.loc[chart_data["offense_team"] == team_to_show]

# Remove rows that are missing a value needed for this chart.
chart_data = chart_data.dropna(subset=[x_feature, y_feature, color_feature])

if chart_data.empty:
    print("No plays match that choice. Check the spelling or choose All teams.")
else:
    # A sample is a smaller random selection from the full table.
    chart_sample = chart_data.sample(n=min(4000, len(chart_data)), random_state=7)

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        data=chart_sample,
        x=x_feature,
        y=y_feature,
        hue=color_feature,
        alpha=0.45,
        ax=ax,
    )
    ax.set_title(f"{y_feature.replace('_', ' ').title()} by {x_feature.replace('_', ' ').title()}")
    ax.set_xlabel(x_feature.replace("_", " ").title())
    ax.set_ylabel(y_feature.replace("_", " ").title())
    ax.legend(title=color_feature.replace("_", " ").title(), bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Step 4: Make a chart that answers your question

Choose a question and change the chart choices to help answer it. You can change the **team**, the **horizontal axis**, the **vertical axis**, or the **colors**. You can also change the chart type if you want.

### Need a starting idea?

- **Do runs and passes tend to gain different yards?** Use `play_type` for color and `yards_gained` for the vertical axis.
- **Does the relationship between distance to a first down and yards gained look different by play type?** Use `yards_to_go` horizontally and `yards_gained` vertically.
- **Does performance look different by quarter?** Put `quarter` on one axis or use it for color.
- **Do teams choose differently when they are ahead or behind?** Try `score_difference_at_start` on the horizontal axis and color by `play_type`.
- **What is the difference between average yards and average EPA?** Change the vertical axis to `epa`.

There is no single right answer. The goal is to make one clear visual and explain what you noticed.

## Step 5: Tell the story

When you present your chart, be ready to say:

1. **My question was…**
2. **My chart shows…**
3. **One thing I noticed was…**

Optional: If you want to save your finished chart as an image, run the cell below. It will make a file you can download from Colab's file panel. A screenshot also works.

In [ ]:
# Optional: save the most recent chart to a picture file.
# If you are using Colab, the download prompt will appear automatically.
try:
    fig.savefig("my_college_football_chart.png", dpi=200, bbox_inches="tight")
    from google.colab import files
    files.download("my_college_football_chart.png")
except ImportError:
    print("Saved as my_college_football_chart.png in the current folder.")

### Thanks for taking part!

You made a chart from real college-football data. You do not need to remember the code after today—the point was to try an idea, make something, and share what you found.